# Further time-step comparisons

Extract CCR_Resolution.zip into /user/cbaker4/connectome and open this notebook there. Reuse the existing data, ccr_python.json and .ccr-venv. This adds new files and a new output folder. Use Run All once.

The plan has 530 trials: four exact replay checks; 46 trials for the previously selected high-weight cases at 0.0125 and 0.00625 ms; and 480 default-setting trials at 0.05 and 0.025 ms, covering all 30 original seeds and eight conditions including active29.

Your existing allocation can be used if it has sufficient time left. The runner reads Slurm's remaining time, limits workers to 3.5 hours or 30 minutes before the allocation ends, whichever is sooner, and refuses to start with less than one hour remaining. A new allocation can use 24 cores, 192000 MB, six hours, no GPU, UB-HPC, smuldoon, general-compute and its matching QOS. Leave modules and node features blank. Up to 16 workers run concurrently; full runtime is not yet measured.

A replay mismatch stops the later stages. Smaller-step results are collected without declaring convergence from an arbitrary cutoff. This remains a study of previously used seeds and fixed comparison sets.


In [ ]:
from pathlib import Path
import os, json, sys, subprocess, time, hashlib
ROOT = Path.cwd().resolve()
STUDY = ROOT / 'results/expanded_sensitivity'
OUT = ROOT / 'results/resolution_20260927'
PYTHON = ROOT / '.ccr-venv/bin/python'
SCRIPT = ROOT / 'scripts/pcdr_resolution.py'
RUN_HOURS = 3.5
LOG = ROOT / (OUT.name + '_controller.log')
if not (ROOT / 'transfer_manifest.json').is_file() or not (STUDY / 'jobs.json').is_file() or not PYTHON.is_file():
    raise RuntimeError('Open this notebook inside the existing /user/cbaker4/connectome directory.')
if hashlib.sha256((ROOT / 'resolution_plan.json').read_bytes()).hexdigest() != '2ca7f3cd2378b4fdd01291eaac15e253d2d5e847cd03d82f42e3017b3ad91d58':
    raise RuntimeError('Follow-up plan differs from the reviewed package.')
plan = json.loads((ROOT / 'resolution_plan.json').read_text())
for name, expected in plan['sources'].items():
    if hashlib.sha256((ROOT / name).read_bytes()).hexdigest() != expected:
        raise RuntimeError('Changed follow-up source: ' + name)
config = json.loads((ROOT / 'ccr_python.json').read_text())
RUN_ENV = os.environ.copy()
for key, value in config.get('env', {}).items():
    if value is None:
        RUN_ENV.pop(key, None)
    else:
        RUN_ENV[key] = value
for key in ['PYTHONHOME', 'PYTHONPATH']:
    RUN_ENV.pop(key, None)
RUN_ENV.update(PYTHONNOUSERSITE='1', MPLBACKEND='Agg', OMP_NUM_THREADS='1', OPENBLAS_NUM_THREADS='1', MKL_NUM_THREADS='1')
subprocess.run([str(PYTHON), '-c', 'import sys,numpy,pandas,pyarrow,brian2; assert sys.version_info[:2]==(3,11); print(sys.version)'], env=RUN_ENV, check=True)
print('Original study:', STUDY)
print('New output:', OUT)
print('Trials:', len(plan['jobs']))


The next cell starts a separate process with a file log, then waits for it. Closing the browser does not stop that process while the allocation stays alive. Do not click Run All a second time while it is running. Logs and progress stay on disk. If the allocation ends, the process stops; a new allocation is not automatic.

Existing output folders are never overwritten. If the run fails, keep the error and results ZIP for review. Do not change the Python versions or delete old outputs to make it proceed. A completed run can be reopened and this notebook rerun to show its download location.


In [ ]:
if OUT.exists():
    record = json.loads((OUT / 'progress.json').read_text())
    if record.get('status') != 'complete':
        raise RuntimeError('An existing run is unfinished or failed. Read ' + str(LOG) + ' and ' + str(OUT / 'progress.json') + '; do not launch another copy.')
else:
    command = [str(PYTHON), '-u', str(SCRIPT), 'run', '--study', str(STUDY), '--out', str(OUT), '--hours', str(RUN_HOURS)]
    with LOG.open('x', encoding='utf-8') as stream:
        process = subprocess.Popen(command, cwd=ROOT, env=RUN_ENV, stdout=stream, stderr=subprocess.STDOUT, start_new_session=True)
    print('Controller PID:', process.pid, 'Log:', LOG, flush=True)
    position = 0
    while True:
        with LOG.open(encoding='utf-8', errors='replace') as stream:
            stream.seek(position)
            text = stream.read()
            position = stream.tell()
        if text:
            print(text, end='', flush=True)
        result = process.poll()
        if result is not None:
            with LOG.open(encoding='utf-8', errors='replace') as stream:
                stream.seek(position)
                print(stream.read(), end='', flush=True)
            break
        time.sleep(5)
    if result:
        print('If present, download:', OUT / 'CCR_resolution_results.zip')
        raise RuntimeError('Run stopped. Keep the log and any result ZIP; the original data are unchanged.')
record = json.loads((OUT / 'progress.json').read_text())
if record['status'] != 'complete':
    raise RuntimeError('Collection is not complete.')
print('Finished:', record['completed'], 'trials')
print('Download:', OUT / 'CCR_resolution_results.zip')


To check from a terminal:

```bash
tail -n 20 /user/cbaker4/connectome/resolution_20260927_controller.log
cat /user/cbaker4/connectome/results/resolution_20260927/progress.json
```

When status is complete and the count is 530, download results/resolution_20260927/CCR_resolution_results.zip and resolution_20260927_controller.log. The ZIP now contains the full spike and delivered-event files as well as rates, traces, summaries and worker logs. No separate tar download is needed.

The controller log records preflight errors. A failed worker stops later stages and packages the available evidence when possible. A hard allocation termination can interrupt packaging. Existing partial folders are preserved and are not automatically resumed. Do not rerun over them or delete them to make a run proceed. Closing the browser does not stop the file-logged controller while the allocation remains active.
